#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze


In [0]:
df=spark.table("workspace.bronze.erp_loc_a101")

#Data Transformation

##Data Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,F.trim(col(field.name)))

##Customer ID Cleanup

In [0]:
df=df.withColumn("cid",
                 F.regexp_replace(col("CID"),"-",""))

##Data Normalization

In [0]:
df=df.withColumn("cntry",
                 F.when(col("CNTRY").isin("USA","US"),"United States")
                 .when(col("CNTRY") == "DE","Germany")
                 .when((col("CNTRY") =="") | col("CNTRY").isNull(),"n/a")
                  .otherwise(col("CNTRY")))


##Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name,new_name in RENAME_MAP.items():
    df=df.withColumnRenamed(old_name,new_name)

#Sanity check of Daataframe

In [0]:

df.limit(10).display()

#Writing Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.erp_customer_location")

#Sanity check of Silver Table

In [0]:
%sql
SELECT * FROM silver.erp_customer_location
